<a href="https://colab.research.google.com/github/deva7102bratta/AI-nsdl-close-price/blob/main/NSDL_Price_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from google.colab import drive

In [ ]:
drive.mount("/content/drive")

In [ ]:
PATH = "/content/drive/MyDrive/CDSL_historical_data_Daily_Full_History.csv"

In [ ]:
df = pd.read_csv(PATH)

In [ ]:
df.drop(columns="Exchange", inplace=True)

In [ ]:
print(df.head())
print(df.columns.tolist())
print(df.shape)

In [ ]:
df = df.sort_values('Date').reset_index(drop=True)

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])

In [ ]:
df["PrevClose"] = df["Close"].shift(1)

In [ ]:
df['PrevVolumeMean3'] = df['Volume'].shift(1).rolling(3, min_periods=1).mean()
df['PrevVolumeMean20'] = df['Volume'].shift(1).rolling(20, min_periods=1).mean()

df['RelativeVolume3'] = df['Volume'] / df['PrevVolumeMean3']
df['RelativeVolume20'] = df['Volume'] / df['PrevVolumeMean20']

In [ ]:
df['LogRelativeVolume3'] = np.log1p(df['RelativeVolume3'])
df['LogRelativeVolume20'] = np.log1p(df['RelativeVolume20'])

In [ ]:
df['OpenReturn'] = (df['Open'] - df['PrevClose']) / df['PrevClose']
df['HighReturn'] = (df['High'] - df['PrevClose']) / df['PrevClose']
df['LowReturn'] =  (df['Low'] - df['PrevClose']) / df['PrevClose']
df['TodayReturn'] = (df['Close'] - df['PrevClose']) / df['PrevClose']

In [ ]:
df['PreviousReturn'] = df['TodayReturn'].shift(1)
df['TomorrowReturn'] = df['TodayReturn'].shift(-1)

In [ ]:

features = [
    'OpenReturn',
    'HighReturn',
    'LowReturn',
    'TodayReturn',
    'PreviousReturn',
    'LogRelativeVolume3',
    'LogRelativeVolume20'
]

X = df[features]
y = df['TomorrowReturn']

model_data = pd.concat([X, y], axis=1).dropna()

X = model_data[features]
y = model_data['TomorrowReturn']

In [ ]:

n = len(X)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end]
y_train = y.iloc[:train_end]

X_val = X.iloc[train_end:val_end]
y_val = y.iloc[train_end:val_end]

X_test = X.iloc[val_end:]
y_test = y.iloc[val_end:]

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
w = np.zeros(X_train_scaled.shape[-1]) +1
b = 0.0

print("Weight:", w)
print("Bias:", b)

In [ ]:
y_pred = X_train_scaled @ w + b
y_pred

In [ ]:
error = y_pred - y_train.to_numpy()
error

In [ ]:
squared_error = error**2
squared_error

In [ ]:
mse = np.mean(squared_error)
mse

In [ ]:
# Learning rate
learning_rate = 1

# 1. Make predictions
y_pred = X_train_scaled @ w + b

# 2. Calculate prediction error
error = y_pred - y_train.to_numpy()

# 3. Number of training examples
n = X_train_scaled.shape[0]

# 4. Calculate gradient of weights
dw = (2 / n) * (X_train_scaled.T @ error)

# 5. Calculate gradient of bias
db = (2 / n) * np.sum(error)

# 6. Update weights
w = w - learning_rate * dw

# 7. Update bias
b = b - learning_rate * db

print("Updated weights:", w)
print("Updated bias:", b)

In [ ]:
learning_rate = 0.01
epochs = 1000

for epoch in range(epochs):

    # 1. Prediction
    y_pred = X_train_scaled @ w + b

    # 2. Error
    error = y_pred - y_train.to_numpy()

    # 3. Number of samples
    n = X_train_scaled.shape[0]

    # 4. Gradients
    dw = (2 / n) * (X_train_scaled.T @ error)
    db = (2 / n) * np.sum(error)

    # 5. Update weights and bias
    w = w - learning_rate * dw
    b = b - learning_rate * db

    # 6. Calculate loss
    mse = np.mean(error ** 2)

    if epoch % 100 == 0:
        print(f"Epoch {epoch}: MSE = {mse}")

In [ ]:
print("Updated weights:", w)
print("Updated bias:", b)

In [ ]:
y_val_pred = X_val_scaled @ w + b

val_mse = np.mean(
    (y_val_pred - y_val.to_numpy()) ** 2
)

print("Validation MSE:", val_mse)

In [ ]:
train_rmse = np.sqrt(
    np.mean((X_train_scaled @ w + b - y_train.to_numpy()) ** 2)
)

val_rmse = np.sqrt(
    np.mean((y_val_pred - y_val.to_numpy()) ** 2)
)

print("Training RMSE:", train_rmse)
print("Validation RMSE:", val_rmse)

In [ ]:
val_actual = y_val.to_numpy()

val_mae = np.mean(
    np.abs(y_val_pred - val_actual)
)

directional_accuracy = np.mean(
    np.sign(y_val_pred) == np.sign(val_actual)
)

print("Validation MAE:", val_mae)
print("Validation Directional Accuracy:", directional_accuracy)

In [ ]:
baseline_pred = X_val['TodayReturn'].to_numpy()

baseline_mae = np.mean(
    np.abs(baseline_pred - y_val.to_numpy())
)

baseline_rmse = np.sqrt(
    np.mean((baseline_pred - y_val.to_numpy()) ** 2)
)

baseline_direction = np.mean(
    np.sign(baseline_pred) == np.sign(y_val.to_numpy())
)

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)
print("Baseline Directional Accuracy:", baseline_direction)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.plot(
    y_val.to_numpy(),
    label="Actual"
)

plt.plot(
    y_val_pred,
    label="Predicted"
)

plt.xlabel("Validation Sample")
plt.ylabel("Tomorrow Return")
plt.title("Actual vs Predicted Tomorrow Return")
plt.legend()
plt.show()

In [ ]:
for feature, weight in zip(features, w):
    print(f"{feature}: {weight}")

print("Bias:", b)